# Solar Filament Segmentation Challenge 2026 — Train the final all-data model

Trains the final submission model with the **exact fold-validated recipe** on **all 707
training photographs** (the development recipe trained on 417; only the training-set
membership differs). Requires a GPU (~5-9 h for 16,384 optimizer updates).

Uses only the competition training polygons — no external solar data, pretrained solar
models, or other ground-truth metadata.

In [ ]:
from pathlib import Path
import json, sys, os, random, time, shutil, hashlib

def _find(name):
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_file()]
    assert hits, f'missing {name}'
    return hits[0]

ANN         = _find('MAGFiLO_1.0_Annotations_kaggle2026_train.json')
TRAIN_ROOT  = ANN.parent.parent                  # the folder containing train/train_images
TEST_IMAGES = next(d for d in Path('/kaggle/input').rglob('test_images')
                   if d.is_dir() and len(list(d.glob('*.jpeg'))) == 180)
OUTPUT      = Path('/kaggle/working')

SRC_DIR = Path('/kaggle/working/src')
if not (SRC_DIR / 'solar_filament').exists():
    pkg = next(p for p in Path('/kaggle/input').rglob('artifacts.py')
               if p.parent.name == 'solar_filament')
    SRC_DIR.mkdir(exist_ok=True)
    shutil.copytree(pkg.parent, SRC_DIR / 'solar_filament')
sys.path.insert(0, str(SRC_DIR))
print('annotations:', ANN)

In [ ]:
# --- dependencies (pinned) ---------------------------------------------------
import importlib.metadata as im, subprocess
def ensure(pkg, version):
    try:
        if im.version(pkg) == version:
            return
    except im.PackageNotFoundError:
        pass
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{pkg}=={version}'], check=True)
ensure('segmentation-models-pytorch', '0.5.0')
ensure('timm', '1.0.30')

import numpy as np
import torch
from torch.utils.data import DataLoader
from solar_filament.artifacts import source_identity
from solar_filament.catalog import AnnotationCatalog
from solar_filament.config import TrainingConfig
from solar_filament.dataset import TrainingCrops, save_training_preview
from solar_filament.experiment import checkpoint_write, optimizer_update
from solar_filament.network import build_network, segmentation_loss
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# --- configuration: the validated recipe -------------------------------------
CONFIG = TrainingConfig(
    encoder='tu-convnext_tiny.fb_in1k', encoder_weights='imagenet',
    crop_size=512, batch_size=2, learning_rate=2e-4, weight_decay=1e-4,
    max_steps=16384, seed=2026, validation_fold=1,   # unused: all-data run
    device='cuda', fixed_crops=0, augment=True, positive_crop_probability=0.95,
)
torch.set_num_threads(4)
random.seed(CONFIG.seed); np.random.seed(CONFIG.seed)
torch.manual_seed(CONFIG.seed); torch.cuda.manual_seed_all(CONFIG.seed)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
device = torch.device('cuda')

catalog = AnnotationCatalog(ANN)
training_names = sorted(catalog.by_filename)
print(f'training photographs: {len(training_names)}')

In [ ]:
# --- training loop (verbatim from the frozen trainer, minus the fold split) ---
run = OUTPUT / 'alldata-convnext-16384-001'
run.mkdir(exist_ok=True)
metadata = {
    'schema_version': 1, 'run_name': run.name, 'status': 'RUNNING',
    'config': CONFIG.to_dict(),
    'source_sha256': source_identity(SRC_DIR / 'solar_filament'),
    'annotation_sha256': catalog.annotation_sha256,
    'training_images': training_names, 'validation_images': [],
    'policy': 'all-photographs final-model run; no validation split, no holdout',
}
network = build_network(CONFIG).to(device)
optimizer = torch.optim.AdamW(network.parameters(), lr=CONFIG.learning_rate,
                              weight_decay=CONFIG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG.max_steps, eta_min=CONFIG.learning_rate * 0.05)
scaler = torch.amp.GradScaler('cuda', enabled=True)
crops = TrainingCrops(TRAIN_ROOT, catalog, training_names, CONFIG)
save_training_preview(crops, run / 'training_preview.png')
loader = DataLoader(crops, batch_size=CONFIG.batch_size, shuffle=False, num_workers=0)

started, losses, overflows, consecutive = time.perf_counter(), [], 0, 0
max_overflows = max(4, CONFIG.max_steps // 100)
with (run / 'training.jsonl').open('x', encoding='utf-8') as stream:
    for step, (images, targets) in enumerate(loader, start=1):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', enabled=True):
            loss = segmentation_loss(network(images), targets, CONFIG.boundary_weight)
        lr = optimizer.param_groups[0]['lr']
        update = optimizer_update(loss, network, optimizer, scaler)
        if update['optimizer_stepped']:
            scheduler.step()
        overflows += int(update['amp_overflow'])
        consecutive = consecutive + 1 if update['amp_overflow'] else 0
        losses.append(float(loss.detach()))
        stream.write(json.dumps({'step': step, 'loss': losses[-1], 'learning_rate': lr, **update,
                                 'elapsed_seconds': time.perf_counter() - started}) + '\n')
        stream.flush()
        if consecutive > 3 or overflows > max_overflows:
            raise FloatingPointError(f'AMP overflow budget exceeded at batch {step}')
        if step == 1 or step % 128 == 0 or step == CONFIG.max_steps:
            print(f'step {step} loss {losses[-1]:.4f} elapsed {time.perf_counter()-started:.0f}s', flush=True)
        if step % 256 == 0 or step == CONFIG.max_steps:
            checkpoint_write(run / 'checkpoint.pt', {
                'model_state': network.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
                'torch_rng_state': torch.get_rng_state(),
                'step': step, 'metadata': metadata})
print('TRAINING COMPLETE:', len(losses), 'steps,', overflows, 'overflows',
      f'{time.perf_counter()-started:.0f}s')

In [ ]:
# --- save final checkpoint + hash -------------------------------------------
checkpoint_write(run / 'checkpoint.pt', {
    'model_state': network.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'scheduler_state': scheduler.state_dict(),
    'scaler_state': scaler.state_dict(),
    'torch_rng_state': torch.get_rng_state(),
    'step': CONFIG.max_steps, 'metadata': {**metadata, 'status': 'COMPLETE'}})
h = hashlib.sha256((run / 'checkpoint.pt').read_bytes()).hexdigest()
(run / 'checkpoint_sha256.txt').write_text(h)
print('checkpoint sha256:', h)